In [1]:

import sys
sys.path.append("../src")

import asyncio

from connection_util import DatabaseConnection
from validation_util import Validation
from savepoints_util import SavePoints
from etl_pipeline import run_etl_pipeline






In [7]:

SCHEMA_NAMES = ('akwaaba_schema','buffalo_schema','colobus_schema')

CLEAR_CACHE = True

In [8]:


async def main():
    try:

        await SavePoints.clear_all_progress(is_clear_cache = CLEAR_CACHE)

        # connect source and target database
        source_conn,target_conn = await asyncio.gather(
            DatabaseConnection.connect_to_source_postgis_database(),
            DatabaseConnection.connect_to_target_postgis_database()
        )

        if not source_conn or not target_conn:
            print("connection failed")
            return 

        #validate schema structure:layer names and column names
        akwaaba_val_result,buffalo_val_result,colobus_val_result = await asyncio.gather(
            Validation.validate_schema(source_conn,SCHEMA_NAMES[0]),
            Validation.validate_schema(source_conn,SCHEMA_NAMES[1]),
            Validation.validate_schema(source_conn,SCHEMA_NAMES[2])
        )

        if not akwaaba_val_result or not buffalo_val_result or not colobus_val_result:
            print("validation failed failed")
            return


        # Extract, transform and load from Group Planning into Chameleon database
        await asyncio.gather(
            run_etl_pipeline(SCHEMA_NAMES[0],source_conn,target_conn),
            run_etl_pipeline(SCHEMA_NAMES[1],source_conn,target_conn),
            run_etl_pipeline(SCHEMA_NAMES[2],source_conn,target_conn)
        )

        print('All Layers are Loaded')
        
    except Exception as e:
        print("ETL Failed with error message",e)


if __name__ == "__main__":
    await main()


Source database connection successful!
Target database connection successful!
Schema validation passed!
Schema validation passed!
Schema validation passed!
All geometries valid: True
All geometries valid: True
All geometries valid: True
All geometries valid: True
All geometries valid: True
Successfully saved 63 records to akwaaba_schema.compartments
akwaaba_schema compartments Loaded Successfully
All geometries valid: True
Successfully saved 12 records to akwaaba_schema.leased_areas
akwaaba_schema leased Areas Loaded Successfully
All geometries valid: True
All geometries valid: True
All geometries valid: True
Successfully saved 743 records to akwaaba_schema.subcompartments
akwaaba_schema Subcompartments Loaded Successfully
All geometries valid: True
All geometries valid: True
All geometries valid: True
All geometries valid: True
All geometries valid: True
All geometries valid: True
Successfully saved 641 records to akwaaba_schema.completed_psps
akwaaba_schema Completed Psps Loaded Succ